# CS5480: Lightweight QA Evaluation with FLAN-T5

## Problem Overview
Large Language Models often hallucinate because they optimize for plausible text rather than factual correctness. This notebook establishes a closed-book baseline for question answering, which will later be compared against retrieval-augmented approaches.

The goal is to measure how well a model can answer questions without external context.

### 1. Imports

In [1]:
# !pip install pandas numpy transformers rank-bm25 collections sentence-transformers

import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rank_bm25 import BM25Okapi
from collections import Counter
from sentence_transformers import SentenceTransformer

## Dataset Description
The dataset consists of:
- `question`: input query
- `short_answers`: ground truth answer
- `long_answers`:

This format enables straightforward evaluation using exact string matching and token overlap metrics.

### 2. Load Dataset Function

In [2]:
df = pd.read_csv("./Natural-Questions-Filtered.csv")

QUESTIONS = df["question"].tolist()
GROUND_TRUTH = df["short_answers"].tolist()
CORPUS = df["long_answers"].tolist()

### 3. Load Model

In [3]:
model_name = "google/flan-t5-small"
TOKENIZER = AutoTokenizer.from_pretrained(model_name)
MODEL = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


### 4. Define Analysis Functions

In [4]:
def generate_answer(question):
    prompt = f"Answer the question: {question}"

    inputs = TOKENIZER(prompt, return_tensors="pt", truncation=True)
    outputs = MODEL.generate(**inputs, max_length=64)

    return TOKENIZER.decode(outputs[0], skip_special_tokens=True)

def exact_match(pred, truth):
    return int(pred.strip().lower() == truth.strip().lower())

def f1_score(pred, truth):
    pred_tokens = pred.lower().split()
    truth_tokens = truth.lower().split()

    pred_counts = Counter(pred_tokens)
    truth_counts = Counter(truth_tokens)

    common = sum((pred_counts & truth_counts).values())

    if common == 0:
        return 0.0

    precision = common/len(pred_tokens)
    recall = common/len(truth_tokens)

    return 2 * (precision * recall) / (precision + recall)

### 5. Define Experiment Loop

In [5]:
def run_experiment(retrieve_fn=None, k=3, limit=100):
    em_scores, f1_scores = [], []

    for i in range(min(limit, len(QUESTIONS))):
        q = QUESTIONS[i]
        truth = GROUND_TRUTH[i]

        # retrieval step
        if retrieve_fn is not None:
            contexts = retrieve_fn(q, k)   # returns list of passages
            context_str = " ".join(contexts)
            prompt = f"Answer the question using the context:\n{context_str}\n\nQuestion: {q}"
        else:
            prompt = f"Answer the question: {q}"

        # generation
        inputs = TOKENIZER(prompt, return_tensors="pt", truncation=True)
        outputs = MODEL.generate(**inputs, max_length=64)
        pred = TOKENIZER.decode(outputs[0], skip_special_tokens=True)

        # evaluation
        em_scores.append(exact_match(pred, truth))
        f1_scores.append(f1_score(pred, truth))

    return np.mean(em_scores), np.mean(f1_scores)

### 6. Find Baseline Performance
We will identify the baseline performance by prompting our model with no regression techniques applied. This way, we can determine if BM25, dense, or hybrid retrieval actually improve model performance.

In [6]:
print("Running baseline...")
em, f1 = run_experiment() # no retrieve_fn

print("\n=== BASELINE RESULTS ===")
print(f"Exact Match Score: {em:.4f}")
print(f"F1 Score: {f1:.4f}")

Running baseline...

=== BASELINE RESULTS ===
Exact Match Score: 0.0200
F1 Score: 0.0670


### 7. Find BM25 Performance

**First, we'll build an index for the answer corpus**

In [7]:
TOKENIZED_CORPUS = [doc.split() for doc in CORPUS]
BM25 = BM25Okapi(TOKENIZED_CORPUS)

**Then, we'll define the retrieval function**

In [8]:
def bm25_retrieve(query, k=3):
    tokenized_query = query.split()

    scores = BM25.get_scores(tokenized_query)
    top_k_idx = np.argsort(scores)[-k:][::-1]

    return [CORPUS[i] for i in top_k_idx]

**Finally, we find the BM25 performance.**

In [9]:
print("Running BM25 experiment...")
em, f1 = run_experiment(retrieve_fn=bm25_retrieve)

print("\n=== BM25 RESULTS ===")
print(f"Exact Match Score: {em:.4f}")
print(f"F1 Score: {f1:.4f}")

Running BM25 experiment...

=== BM25 RESULTS ===
Exact Match Score: 0.0300
F1 Score: 0.0879


### 8. Find Dense Retrieval Performance

**First, we build the embeddings for dense retrieval**

In [ ]:
EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2")

CORPUS_EMBEDDINGS = EMBED_MODEL.encode(
    CORPUS,
    batch_size=32,
    convert_to_numpy=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Then, we define the retrieval function**

In [ ]:
def dense_retrieve(query, k=3):
    query_emb = EMBED_MODEL.encode(query, convert_to_numpy=True)

    scores = np.dot(CORPUS_EMBEDDINGS, query_emb)
    top_k_idx = np.argsort(scores)[-k:][::-1]

    return [CORPUS[i] for i in top_k_idx]

**Finally, we'll run the experiment**

In [ ]:
print("Running dense experiment...")
em, f1 = run_experiment(retrieve_fn=dense_retrieve)

print("\n=== DENSE RESULTS ===")
print(f"Exact Match Score: {em:.4f}")
print(f"F1 Score: {f1:.4f}")